# Gemma 4 Developer Agent package

This competition notebook writes a validated agent ZIP to `/kaggle/working/submission.zip`, with `agent.yaml` at the archive root. It embeds only the selected agent files; it does not use local issue solutions, hidden labels, or answer patches. The competition scoring harness supplies the required Gemma 4 model when it evaluates the ZIP.

Run all cells, then submit this notebook version's generated ZIP.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
COMP = Path('/kaggle/input/competitions/gemma-4-developer-agent')
print('Working directory:', WORK)
print('Python:', platform.python_version())
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, capture_output=True, timeout=10)
    print('Visible GPUs:', gpu.stdout.strip() or gpu.stderr.strip() or 'none')
except Exception as exc:
    print('GPU inventory unavailable:', type(exc).__name__)
tasks = COMP / 'tasks.jsonl'
if tasks.is_file():
    with tasks.open(encoding='utf-8') as stream:
        count = sum(1 for line in stream if line.strip())
    print('Public task rows:', count)
else:
    print('Public tasks.jsonl is not mounted; ZIP build does not depend on it.')


## Build a clean, root-layout submission archive

The files below are embedded from the selected agent directory. The ZIP writer uses only that explicit file map, so notebook outputs, logs, training scripts, or inspection wheels cannot leak into the competition archive.


In [ ]:
FILES = {'agent.yaml': 'name: gemma4_swe_agent_v2\n'
               'model: gemma-4-31b-it-qat-w4a16-ct\n'
               'description: Autonomous software engineer that localizes and repairs repository issues with '
               'concise evidence checks.\n'
               'instruction: !include prompts/system.md\n'
               'tools:\n'
               '  - run_command\n'
               '  - get_status\n'
               '  - read_file\n'
               '  - edit_file\n'
               '  - write_file\n'
               '  - submit_patch\n'
               '  - agent_tool:\n'
               '      config_path: sub_agents/code_analyzer.yaml\n'
               '      skip_summarization: true\n'
               'generate_content_config: !include configs/sampling.yaml\n',
 'configs/sampling.yaml': 'temperature: 0.2\n'
                          'top_p: 0.95\n'
                          'top_k: 40\n'
                          'max_output_tokens: 4096\n'
                          'thinking_config:\n'
                          '  thinking_budget: 2048\n'
                          '  include_thoughts: false\n',
 'eval_config.yaml': 'evaluation:\n'
                     '  timeout_seconds: 120\n'
                     '  max_tool_calls: 28\n'
                     '  max_time_minutes: 8\n'
                     '  max_turns: 24\n',
 'prompts/system.md': 'You are an autonomous senior software engineer fixing one issue in the repository '
                      'mounted at `/workspace`.\n'
                      '\n'
                      '## Required finish\n'
                      'The task is not complete until you invoke `submit_patch()` as your final tool call. A '
                      'written explanation does not submit a patch. Make the strongest evidence-based '
                      'production-code change you can and submit it even when a test cannot run or still '
                      'fails. Do not finish with an empty patch merely because verification is incomplete.\n'
                      '\n'
                      '## Tests are strictly read-only\n'
                      'Never call `edit_file` or `write_file` on a test path. This includes any path under '
                      '`tests/`, any `test_*.py` or `*_test.py` file, `conftest.py`, and `pytest.ini`. Do '
                      'not create, delete, rename, or change tests or fixtures, even to reproduce behavior. '
                      'Before submission, inspect the changed-path list. If a test path was changed, run '
                      '`git restore -- <that exact test path>` and recheck the diff; submit production code '
                      'only. The benchmark applies its own tests after you finish.\n'
                      '\n'
                      '## Work quickly from evidence\n'
                      '1. Use the supplied workspace layout and issue. If the issue names a file, symbol, or '
                      'error string, inspect that location first; otherwise run one bounded `rg --files` or '
                      'literal `rg -n -F` search.\n'
                      '2. Read the smallest useful implementation range and form one concrete failure '
                      'hypothesis. Do not search git history. Do not run tests before the first edit unless '
                      'a tiny reproducer is already available.\n'
                      '3. If two quick source checks do not identify a plausible location, call '
                      '`code_analyzer` once with the issue and evidence. It is read-only; confirm its '
                      'suggestion in source.\n'
                      '4. Make one minimal production-code edit by about the eighth model/tool step. Use '
                      '`edit_file(filepath, old_string, new_string)` with text copied exactly from source. '
                      'Retry once after rereading a smaller range if it fails. Never modify tests, fixtures, '
                      '`conftest.py`, `pytest.ini`, packaging, or runner configuration.\n'
                      '5. Inspect the diff and run at most one short, relevant check. If that check finds a '
                      'repairable error, make one focused correction. Never run a full suite, repeat an '
                      'unchanged check, install packages, or use the network.\n'
                      '6. Inspect the changed paths and restore any test-path changes using `git restore -- '
                      '<exact path>`. Confirm only intended production files remain.\n'
                      '7. Call `submit_patch()` no later than model/tool step 18 or when 60 seconds remain, '
                      'whichever comes first. Do not make another tool call after it. If unsure, submit the '
                      'best coherent source patch rather than spending the remaining budget on more '
                      'searches.\n'
                      '\n'
                      '## Repository navigation\n'
                      '- Prefer `rg --files` and literal `rg -n -F` searches; keep output bounded.\n'
                      '- Use graph tools only if they are available and a source-level symbol has already '
                      'been identified. Pass exact symbol names, not prose. If graph data is missing, '
                      'continue with source search and do not repeat the same graph query.\n'
                      '- Avoid broad directory dumps, repeated reads, and large file rewrites. Keep tool '
                      'payloads concise. Check `get_status` only when the remaining budget could change the '
                      'next action.\n'
                      '\n'
                      '## Repair quality\n'
                      "- Preserve public interfaces and nearby conventions; handle the issue's stated edge "
                      'cases.\n'
                      '- Never invent verification results. Check the final diff before submitting.\n'
                      '- A focused check failure is evidence for at most one correction; it is not a reason '
                      'to delay `submit_patch()`.\n',
 'sub_agents/analyzer.md': 'You are a read-only code localizer for one repository issue. Use the task '
                           'statement and current source only. Return at most 180 words with: likely '
                           'implementation files and exact symbols; the source evidence supporting the '
                           'location; one useful existing test or narrow check; and remaining uncertainty. '
                           'Start with bounded literal `rg` searches and short file reads. Do not edit '
                           'files, run broad tests, inspect git history, or claim a repair is verified. If '
                           'graph data is unavailable, continue with ordinary source inspection.\n',
 'sub_agents/code_analyzer.yaml': 'name: code_analyzer\n'
                                  'model: gemma-4-31b-it-qat-w4a16-ct\n'
                                  'description: Read-only repository localizer that reports likely '
                                  'implementation symbols and a narrow verification target.\n'
                                  'instruction: !include analyzer.md\n'
                                  'tools:\n'
                                  '  - run_command\n'
                                  '  - read_file\n'
                                  'generate_content_config:\n'
                                  '  temperature: 0.1\n'
                                  '  top_p: 0.95\n'
                                  '  max_output_tokens: 1536\n'
                                  '  thinking_config:\n'
                                  '    include_thoughts: false\n'}


In [ ]:
import re, zipfile
from pathlib import PurePosixPath

EXPECTED_MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
ALLOWED = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}
assert 'agent.yaml' in FILES and len([p for p in FILES if p in {'agent.yaml','agent.yml','root_agent.yaml','root_agent.yml'}]) == 1
for rel, content in FILES.items():
    path = PurePosixPath(rel)
    assert not path.is_absolute() and '..' not in path.parts, rel
    assert path.suffix.lower() in ALLOWED, rel
    if path.suffix.lower() == '.py':
        assert 'skills' in path.parts, rel
    if path.suffix.lower() in {'.yaml', '.yml'}:
        for model in re.findall(r'^\s*model:\s*([^\s#]+)', content, re.M):
            assert model == EXPECTED_MODEL, (rel, model)
        for included in re.findall(r'!include\s+([^\s#]+)', content):
            target = (PurePosixPath(rel).parent / included)
            target = PurePosixPath(*[p for p in target.parts if p not in {'.', ''}])
            assert '..' not in target.parts and target.as_posix() in FILES, (rel, included)

zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for rel, content in sorted(FILES.items()):
        archive.writestr(rel, content.encode('utf-8'))
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
    assert 'agent.yaml' in names and not any(n.startswith('../') for n in names)
    assert archive.testzip() is None
    assert set(names) == set(FILES)
print('Created:', zip_path)
print('Archive bytes:', zip_path.stat().st_size)
print('Root agent present:', 'agent.yaml' in names)
print('Files:', len(names))


## Next step

Submit this notebook version's `submission.zip` as the competition artifact. The package uses the competition's required Gemma 4 checkpoint and includes no adapter.
